# ДЗ 1 — Побить линейную регрессию на House Prices

> Если зачем-то закрыли лекцию: это [Модуль 3](https://itrubnikov.github.io/Train_of_Thought/modules/03-catboost) курса «От нуля до своих агентов».

Цель — за вечер пройти полный пайплайн на классике Kaggle. Грузим **Ames Housing** (датасет [House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)): ~1500 домов в городе Эймс, Айова, 80 фичей (год постройки, район, материал крыши, площадь подвала и так далее), нужно предсказать цену продажи. Это ровно та задача, которую мы разбирали в ментальной модели лекции: «дерево №1 говорит 10 млн, дерево №2 чинит ошибку…».

Обучаем линейную регрессию-бейзлайн, обгоняем её CatBoost'ом **без feature engineering'а**, сравниваем графически (line vs function), смотрим SHAP-объяснения.

**Что от вас требуется:** заполнить 3 блока `TODO`. Остальное уже написано. CatBoost и SHAP установятся первой ячейкой.

**Время:** 45—60 минут.

**Как сдавать:** `Файл → Сохранить копию на Диске` → дописать `TODO` → запустить все ячейки → `Поделиться → у кого есть ссылка → Просмотр` → прислать ссылку в чат курса как `[Модуль 3, ДЗ 1] {ссылка}`.

## Шаг 0. Установка библиотек

В Colab нужны только два пакета сверху (`pandas`, `scikit-learn`, `matplotlib` уже стоят).

In [ ]:
!pip install -q catboost shap && echo "[ok] catboost и shap установлены"

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Подтверждение, что окружение собралось — иначе ячейка молчит и кажется,
# будто она ничего не сделала (на самом деле просто у import'ов нет вывода).
print(f'pandas {pd.__version__} | numpy {np.__version__} | окружение готово')

## Шаг 1. Данные — House Prices (Ames Housing)

Самая ходовая Kaggle-классика для табличного ML: 1 460 домов в городе Эймс, штат Айова, продажи 2006—2010 годов. 80 признаков на дом — от площади гаража до качества кухни. Таргет — цена продажи `SalePrice` в долларах.

### Как подключить датасет (один раз перед запуском)

Дальнейший код пытается найти `train.csv` тремя способами по убыванию надёжности:

1. **Kaggle с подключённым competition** — без интернета, мгновенно. **Рекомендуется на Kaggle.**
2. **OpenML через `fetch_openml`** — для Colab/локально/Kaggle с Internet=On. Иногда падает по 504, потому что OpenML — некоммерческий сервис.
3. Если оба не сработали — печатается понятная ошибка с инструкцией.

**Если вы в Kaggle, сделайте 2 шага:**

1. Справа в ноутбуке откройте панель **Notebook** → секция **Input** → кнопка **`+ Add Input`**.
2. В открывшемся окне переключите вкладку на **«Competitions»** (она по умолчанию на «Datasets»). В поиске наберите `House Prices` и выберите карточку **«House Prices — Advanced Regression Techniques»** (Ongoing, 5132 Teams, Getting Started). Жмите **`+`**.

Данные смонтируются в работающий kernel сразу — рестарт обычно не нужен. Следующая ячейка найдёт `train.csv` сама — на любом из layout'ов (`/kaggle/input/<slug>/train.csv` или `/kaggle/input/competitions/<slug>/train.csv`). Если `/kaggle/input/` после подключения почему-то пуст, сделайте `Run → Restart Session` и снова сверху.

**Если вы в Colab или локально** — ничего не делайте, код сразу пойдёт через OpenML.

In [ ]:
import glob
from pathlib import Path

# Универсальный поиск train.csv в /kaggle/input/.
# Kaggle монтирует competition'ы либо плоско (`/kaggle/input/<slug>/`), либо
# вложенно (`/kaggle/input/competitions/<slug>/`) — glob ловит оба варианта.
kaggle_candidates = glob.glob('/kaggle/input/**/train.csv', recursive=True)
kaggle_candidates = [c for c in kaggle_candidates if 'house' in c.lower()]

if kaggle_candidates:
    df = pd.read_csv(kaggle_candidates[0])
    print(f'[ok] Загружено из Kaggle Input: {kaggle_candidates[0]} ({df.shape})')
else:
    try:
        ames = fetch_openml(name='house_prices', version=1, as_frame=True, parser='auto')
        df = ames.frame.copy()
        print(f'[ok] Загружено через OpenML: {df.shape}')
    except Exception as e:
        raise RuntimeError(
            'Не получилось скачать датасет. На Kaggle: подключите competition '
            '(правая панель → Add Input → Competitions → "House Prices - '
            'Advanced Regression Techniques") и сделайте Run → Restart Session. '
            'Либо включите Internet (Notebook Settings → Internet → On). '
            f'Исходная ошибка OpenML: {e}'
        )

if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print(df.shape)
df[['LotArea', 'YearBuilt', 'OverallQual', 'Neighborhood', 'GrLivArea', 'SalePrice']].head()

### Что в этой таблице

Каждая **строка** — это один проданный дом (всего 1 460). Каждая
**колонка** — одна характеристика дома. Всего их 80 плюс цена.

Вот что значат колонки в превью выше:

| Колонка | Что это | Тип |
| --- | --- | --- |
| `LotArea` | площадь участка в кв. футах | число |
| `YearBuilt` | год постройки | число |
| `OverallQual` | общая оценка качества отделки, 1—10 | число |
| `Neighborhood` | район города (CollgCr, Veenker, …) | **категория** |
| `GrLivArea` | жилая площадь над землёй, кв. футы | число |
| `SalePrice` | **цена продажи в долларах — это таргет**, то, что учимся предсказывать | число |

`(1460, 80)` в выводе означает: 1 460 строк (домов) и 80 колонок-фичей
(после того как мы выкинули служебную колонку `Id`). Остальные 74
колонки, которых не видно в превью, — это всё про дом: тип крыши,
материал стен, площадь гаража, есть ли бассейн, и т.д. Полное описание
каждой — в файле `data_description.txt` из competition.

Дальше мы отделим `SalePrice` (ответ) от остальных колонок (вопрос) и
будем учить модель по 80 фичам предсказывать цену.

In [ ]:
# Таргет — log(SalePrice). Цены ходят от $35k до $755k, разброс на порядок —
# на лог-шкале RMSE интерпретируется как «относительная ошибка», и так
# меряют этот датасет на самой Kaggle-лидерборде.
# Не используем df.pop(): он удаляет колонку из df, и при повторном
# запуске ячейки SalePrice уже нет -> KeyError. drop() не трогает df.
y = np.log1p(df['SalePrice'])
X = df.drop(columns=['SalePrice'])

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()
print(f'категориальных: {len(cat_features)}')
print(f'числовых: {len(num_features)}')
print(f'таргет log(SalePrice): mean={y.mean():.2f}, std={y.std():.2f}')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## Шаг 2. Бейзлайн — линейная регрессия (Ridge) с one-hot

Готовый код, просто запустите. Это та модель, которую вам нужно обогнать.

Два слова про **что это вообще такое**:

- **Линейная регрессия** — самая простая обучаемая модель для предсказания числа. Берёт все ваши признаки (площадь, год постройки, район…), умножает каждый на свой коэффициент-вес, складывает и выдаёт предсказание. Учится за секунды, объясняется в одну строку, и поэтому это **дефолтный бейзлайн** в табличном ML: если ваша новая навороченная модель не бьёт линейку — она вам не нужна. `Ridge` — это та же линейная регрессия, но с L2-регуляризацией, чтобы не сходила с ума на скоррелированных колонках.
- **One-hot encoding** — способ скормить *текстовую* категорию в модель, которая умеет только числа. Колонка `Neighborhood` со значениями `NAmes / OldTown / CollgCr / ...` (25 районов) превращается в 25 отдельных числовых колонок, в каждой 0 или 1 («это NAmes? — да/нет»). Без этого `Ridge` не знает, что делать со строкой `'NAmes'`. Главная фишка CatBoost'а — что ему всё это руками делать не надо.
- **`StandardScaler`** — приводит числовые колонки к нулевому среднему и единичному разбросу. Линейным моделям это важно, чтобы коэффициенты для «площади в кв.футах» и «года постройки» жили в одном масштабе. Деревьям (CatBoost и компании) — пофиг, они работают порогами и масштабу безразличны.
- **`ColumnTransformer` + `Pipeline`** — стандартный sklearn-приём, чтобы препроцессинг и сама модель ехали как одно целое: `.fit()` сначала обучает преобразования (one-hot, scaler), потом модель; `.predict()` применяет их в том же порядке.
- **RMSE на log(SalePrice)** — метрика, которую мы меряем. На лог-цене ошибка ~0.13 означает «модель промахивается примерно на ±14% от цены». Это та же метрика, что используется на Kaggle-лидерборде этого датасета (RMSLE). 0.16—0.18 — приличный честный бейзлайн от линейки.

### Прежде чем обучить Ridge — посмотрим глазами, что такое «линейная регрессия»

Все эти слова про «умножает признаки на веса и складывает» становятся понятнее на одной картинке. Сначала обучим **сильно упрощённую** версию модели на **одном признаке** — `GrLivArea` (жилая площадь) против цены. Это даст нам **красную прямую**, которую можно нарисовать в 2D и увидеть.

Полный Ridge ниже делает **ровно то же самое**, только в 80-мерном пространстве: вместо одного коэффициента подбирает 80, вместо прямой получается гиперплоскость. Картинку не нарисуешь, но идея та же — найти такую «наклонку», которая минимизирует среднеквадратичную ошибку на всех точках.

In [ ]:
from sklearn.linear_model import LinearRegression

# Упрощённая модель: одна фича вместо 80. y_tr хранится в логе цен —
# для наглядности конвертируем обратно в доллары через expm1.
x_demo = X_tr['GrLivArea'].values.reshape(-1, 1)
y_demo = np.expm1(y_tr.values)

lr_demo = LinearRegression().fit(x_demo, y_demo)
xs = np.linspace(x_demo.min(), x_demo.max(), 100).reshape(-1, 1)
ys = lr_demo.predict(xs)

b = float(lr_demo.intercept_)
k = float(lr_demo.coef_[0])

fig, ax = plt.subplots(figsize=(10, 6))

# Все train-дома облаком
ax.scatter(x_demo, y_demo / 1000, alpha=0.3, s=15,
           label=f'{len(x_demo)} домов из train (Эймс, Айова)')

# Та самая красная прямая, которая и есть «модель»
ax.plot(xs, ys / 1000, 'r-', linewidth=2.5,
        label=f'модель: цена ≈ ${b/1000:.0f}k + ${k:.0f} × площадь')

# Подсветим три конкретных дома с подписями (маленький, средний, большой)
for target_area in [800, 1500, 2800]:
    idx = int(np.abs(x_demo.ravel() - target_area).argmin())
    area = float(x_demo[idx, 0])
    real_price = float(y_demo[idx])
    pred_price = float(lr_demo.predict([[area]])[0])
    ax.scatter([area], [real_price / 1000], s=180, color='orange',
               edgecolor='black', linewidth=2, zorder=5)
    ax.annotate(
        f'{area:.0f} кв.фт\nреальная ${real_price/1000:.0f}k\nмодель ${pred_price/1000:.0f}k',
        xy=(area, real_price / 1000),
        xytext=(area + 250, real_price / 1000 + 50),
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray'),
        arrowprops=dict(arrowstyle='->', color='gray', alpha=0.7),
    )

ax.set_xlabel('GrLivArea (жилая площадь, кв.фт)')
ax.set_ylabel('цена дома')
ax.set_title('Линейная регрессия в одном измерении\nкрасная прямая — это и есть «модель»; полный Ridge ниже — то же самое, но в 80-мерии')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}k'))
plt.tight_layout()
plt.savefig('linear_regression_demo.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'[ok] выучили формулу: цена ≈ ${b/1000:.0f}k + ${k:.0f} за каждый кв.фт')
print(f'     значит каждые дополнительные 100 кв.фт ≈ +${k*100/1000:.0f}k к цене')
print(f'     дом 1000 кв.фт ≈ ${(b + k*1000)/1000:.0f}k, дом 2500 кв.фт ≈ ${(b + k*2500)/1000:.0f}k')

In [ ]:
# Чиним пропуски: для числовых — медиана, для категорий — отдельная метка 'missing'
X_tr_fill = X_tr.copy()
X_te_fill = X_te.copy()
for c in num_features:
    med = X_tr_fill[c].median()
    X_tr_fill[c] = X_tr_fill[c].fillna(med)
    X_te_fill[c] = X_te_fill[c].fillna(med)
for c in cat_features:
    X_tr_fill[c] = X_tr_fill[c].fillna('missing').astype(str)
    X_te_fill[c] = X_te_fill[c].fillna('missing').astype(str)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features),
])
ridge = Pipeline([
    ('prep', preprocess),
    ('reg', Ridge(alpha=10.0, random_state=RANDOM_STATE)),
])

t0 = time.time()
ridge.fit(X_tr_fill, y_tr)
ridge_time = time.time() - t0

ridge_rmse = mean_squared_error(y_te, ridge.predict(X_te_fill)) ** 0.5
print(f'Ridge  RMSE = {ridge_rmse:.4f}  ({ridge_time:.1f} s)')

## Шаг 3. TODO 1 — CatBoost

Обучите `CatBoostRegressor` так, чтобы:
- получить RMSE ≤ 0.135 на тесте (**меньше** = лучше, в отличие от AUC),
- передать `cat_features` ЯВНО (иначе будет беда),
- использовать `eval_set=(X_te, y_te)` и `early_stopping_rounds=50`,
- замерить время обучения в `cat_time`.

CatBoost умеет в пропущенные значения для **числовых** колонок сам, но **в категориальных** NaN не любит — заменим их на `'missing'`.

Подсказка-каркас:

```python
from catboost import CatBoostRegressor

X_tr_cb = X_tr.copy()
X_te_cb = X_te.copy()
for c in cat_features:
    X_tr_cb[c] = X_tr_cb[c].fillna('missing').astype(str)
    X_te_cb[c] = X_te_cb[c].fillna('missing').astype(str)

cat_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    cat_features=cat_features,
    eval_metric='RMSE',
    early_stopping_rounds=50,
    random_seed=RANDOM_STATE,
    verbose=200,
)
t0 = time.time()
cat_model.fit(X_tr_cb, y_tr, eval_set=(X_te_cb, y_te))
cat_time = time.time() - t0
cat_rmse = mean_squared_error(y_te, cat_model.predict(X_te_cb)) ** 0.5
```

Перепишите ячейку под себя и **проверьте**, что RMSE ≤ 0.135. Переменные `cat_model`, `cat_rmse`, `cat_time`, `X_tr_cb`, `X_te_cb` нужны дальше для графиков и SHAP.

In [ ]:
# TODO 1: обучить CatBoostRegressor, сохранить в cat_model, cat_rmse, cat_time, X_tr_cb, X_te_cb
raise NotImplementedError('Реализуй TODO 1 — см. инструкцию выше.')

print(f'CatBoost RMSE = {cat_rmse:.4f}  ({cat_time:.1f} s)')
assert cat_rmse <= 0.135, f'RMSE {cat_rmse:.4f} выше 0.135 — проверь cat_features, iterations и early_stopping_rounds'

In [ ]:
comparison = pd.DataFrame([
    {'model': 'Ridge + OneHot', 'RMSE (log price)': round(ridge_rmse, 4), 'time, s': round(ridge_time, 2)},
    {'model': 'CatBoost',       'RMSE (log price)': round(cat_rmse, 4),   'time, s': round(cat_time, 2)},
])
comparison.sort_values('RMSE (log price)')

## Шаг 4. Сравнение моделей графически

Цифры в табличке — это хорошо, но **картинка показывает разницу нагляднее**. Два графика ниже отвечают на два вопроса.

**График 1 — «насколько модель попадает в настоящую цену».** Scatter: по оси X — настоящая цена дома (log), по оси Y — что предсказала модель. Красная диагональ — идеальное предсказание. Чем плотнее облако к диагонали, тем точнее модель. У Ridge точки заметно разбегаются от диагонали, особенно по краям; у CatBoost облако сжимается вокруг линии.

**График 2 — «линейка vs функция», главное визуальное доказательство.** Берём один «типичный дом» (все фичи на медиане/моде из train), меняем только жилую площадь `GrLivArea` от 500 до 4500 кв.фт, смотрим что предсказывает каждая модель. Ridge **обязан** нарисовать прямую линию — это математика линейной модели (`цена = a + b · площадь + ...`). CatBoost нарисует кривую со ступеньками и изломами — он внутри сотни деревьев-«правил» `если площадь > 2000 и качество > 7, то +Δ`. Эти ступеньки и есть та самая нелинейность, ради которой бустинг и нужен.

In [ ]:
# График 1: предсказание vs реальность для обеих моделей
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, sharex=True)

ridge_preds = ridge.predict(X_te_fill)
cb_preds = cat_model.predict(X_te_cb)

for ax, name, preds in [
    (axes[0], 'Ridge (линейка)', ridge_preds),
    (axes[1], 'CatBoost (ансамбль деревьев)', cb_preds),
]:
    ax.scatter(y_te, preds, alpha=0.4, s=20)
    lo, hi = float(y_te.min()), float(y_te.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.2, label='идеальное предсказание')
    rmse = mean_squared_error(y_te, preds) ** 0.5
    ax.set_title(f'{name}\nRMSE = {rmse:.4f}')
    ax.set_xlabel('настоящая log(цена)')
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('предсказанная log(цена)')
plt.suptitle('Чем плотнее точки к красной диагонали — тем точнее модель', y=1.02)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=120, bbox_inches='tight')
plt.show()
print('[ok] сохранил predicted_vs_actual.png')

In [ ]:
# График 2: «линия vs функция». Фиксируем все фичи на типичных значениях,
# варьируем только GrLivArea — и смотрим форму кривой предсказания.

# «Типичный дом» — медиана числовых + мода категориальных по train
template_num = X_tr.median(numeric_only=True)
template_cat = X_tr[cat_features].mode().iloc[0]
template = pd.concat([template_num, template_cat]).reindex(X_tr.columns)

grliv_range = np.linspace(500, 4500, 120)
synth = pd.DataFrame([template.values] * len(grliv_range), columns=X_tr.columns)
synth['GrLivArea'] = grliv_range

# Готовим для Ridge (нужны fill'ы под scaler/onehot)
synth_ridge = synth.copy()
for c in num_features:
    synth_ridge[c] = pd.to_numeric(synth_ridge[c], errors='coerce').fillna(X_tr[c].median())
for c in cat_features:
    synth_ridge[c] = synth_ridge[c].fillna('missing').astype(str)
ridge_curve = ridge.predict(synth_ridge)

# Готовим для CatBoost (только категории в строки)
synth_cb = synth.copy()
for c in cat_features:
    synth_cb[c] = synth_cb[c].fillna('missing').astype(str)
cb_curve = cat_model.predict(synth_cb)

# Рисуем в долларах для читаемости (np.expm1 — обратная к log1p)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(grliv_range, np.expm1(ridge_curve), label='Ridge — прямая линия', linewidth=2.5)
ax.plot(grliv_range, np.expm1(cb_curve), label='CatBoost — ступеньки и изломы', linewidth=2.5)
ax.set_xlabel('GrLivArea (жилая площадь, кв.фт)')
ax.set_ylabel('предсказанная цена дома')
ax.set_title('Как модель видит зависимость «площадь → цена»\n(все остальные фичи зафиксированы на медиане/моде)')
ax.legend()
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('line_vs_function.png', dpi=120, bbox_inches='tight')
plt.show()
print('[ok] сохранил line_vs_function.png')

## Шаг 5. TODO 2 — SHAP summary plot

Постройте `shap.summary_plot` для CatBoost-модели. Сохраните картинку как `shap_summary.png`.

Что такое SHAP, в одну строку: способ для каждой фичи показать, **насколько и в какую сторону** она в среднем толкает предсказание модели. Для регрессии знак прямой: «+0.10 в log-цене ≈ +10% к стоимости дома».

Каркас:

```python
import shap
explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_te_cb)
shap.summary_plot(shap_values, X_te_cb, max_display=10, show=False)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
```

После запуска ответьте одним предложением (в ячейке ниже): **«топ-3 фичи, которые сильнее всего двигают цену дома, — это …»**. Спойлер: скорее всего там окажутся `OverallQual` (общее качество), `GrLivArea` (жилая площадь) и что-то из района или года постройки.

In [ ]:
# TODO 2: SHAP summary plot для CatBoost-модели
raise NotImplementedError('Реализуй TODO 2 — см. инструкцию выше.')

**Ваш ответ:** топ-3 фичи, которые сильнее всего двигают цену дома, — это … _(допишите)_

## Шаг 6. TODO 3 — Waterfall plot для одного дома

`summary_plot` показал общую картину. Теперь зум на один конкретный дом: возьмите **самый дорогой** дом в тестовой выборке (тот, для которого модель предсказала максимальную цену), постройте для него waterfall plot, сохраните как `shap_waterfall.png` и опишите одной фразой, какие именно фичи задрали его цену.

Каркас:

```python
preds = cat_model.predict(X_te_cb)
idx = int(np.argmax(preds))  # самый дорогой по предсказанию
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_te_cb.iloc[idx],
        feature_names=X_te_cb.columns.tolist(),
    ),
    show=False,
)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'предсказанная цена: ${np.expm1(preds[idx]):,.0f}')
```

In [ ]:
# TODO 3: waterfall plot для самого дорогого дома
raise NotImplementedError('Реализуй TODO 3 — см. инструкцию выше.')

**Ваш ответ:** модель оценила этот дом так дорого, потому что … _(допишите одной фразой)_

## Что вы только что сделали

За один вечер вы прошли путь, который ещё 10 лет назад занимал у ML-инженера неделю: загрузили реальный Kaggle-датасет (классику House Prices), обучили две модели (бейзлайн Ridge и CatBoost), сравнили их **на цифрах (RMSE) и графически** (линейка vs ступеньки), объяснили лучшую через SHAP — глобально и по одному конкретному дому. Это и есть «правая ветка карты ML» из [Модуля 2](https://itrubnikov.github.io/Train_of_Thought/modules/02-ml-map) в действии.

**Чек перед сдачей:**
- [ ] CatBoost RMSE ≤ 0.135 (на log-цене).
- [ ] В таблице сравнения видны цифры обеих моделей.
- [ ] `predicted_vs_actual.png` сохранился — облако CatBoost явно плотнее к диагонали, чем у Ridge.
- [ ] `line_vs_function.png` сохранился — Ridge даёт прямую линию, CatBoost рисует ступенчатую кривую.
- [ ] `shap_summary.png` сохранился и видно осмысленные фичи (`OverallQual`, `GrLivArea`, `Neighborhood`, `YearBuilt` — а не `Id`).
- [ ] `shap_waterfall.png` сохранился и подписан одной фразой.
- [ ] Прислана ссылка в чат курса как `[Модуль 3, ДЗ 1] {ссылка}`.

Дальше — [Модуль 4a: micrograd за час](https://itrubnikov.github.io/Train_of_Thought/modules/04a-micrograd).